# Comparador de ativos da B3

Este notebook compara o desempenho histórico de ações, ETFs e índices ligados ao mercado brasileiro.

## Perguntas analisadas

- Qual ativo apresentou o maior retorno no período?
- Qual teve o melhor retorno ajustado ao risco?
- Qual apresentou a menor volatilidade?
- Qual sofreu a maior perda em relação ao seu pico?
- Como os retornos dos ativos se relacionam?
- Como teria evoluído um investimento inicial em cada ativo?

> **Aviso:** este material possui finalidade educacional e não constitui recomendação de investimento.


## 1. Bibliotecas

O projeto utiliza:

- `yfinance` para baixar preços históricos;
- `pandas` e `NumPy` para tratamento e cálculo;
- `Matplotlib` para visualização;
- `pathlib` para organizar os arquivos gerados.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")


## 2. Configurações

Altere somente esta célula para executar uma nova comparação.

### Observações

- Códigos simples da B3 recebem `.SA` automaticamente.
- Use o símbolo completo para índices ou outros mercados, como `^BVSP`.
- `DATA_FIM = None` busca os dados mais recentes disponíveis.
- A taxa livre de risco é usada nos índices de Sharpe e Sortino.
- Os ativos são comparados no intervalo em que todos possuem dados.


In [ ]:
ATIVOS = ["PETR4", "VALE3", "WEGE3", "BOVA11"]

DATA_INICIO = "2015-01-01"
DATA_FIM = None  # None = dados mais recentes disponíveis

VALOR_INICIAL = 10_000.00
TAXA_LIVRE_RISCO_ANUAL = 0.00

DIAS_NEGOCIACAO_ANO = 252
SALVAR_RESULTADOS = False

PASTA_FIGURAS = Path("reports/figures")
PASTA_TABELAS = Path("reports/tables")


## 3. Preparação dos códigos dos ativos

A função abaixo:

- remove espaços;
- transforma os códigos em maiúsculas;
- adiciona `.SA` aos códigos simples da B3;
- preserva símbolos especiais;
- remove duplicidades;
- exige pelo menos dois ativos.


In [ ]:
def normalizar_ativos(
    ativos: Iterable[str],
    sufixo_b3: str = ".SA",
) -> list[str]:
    """Padroniza códigos para consulta no Yahoo Finance."""

    resultado: list[str] = []

    for ativo_original in ativos:
        ativo = str(ativo_original).strip().upper()

        if not ativo:
            continue

        possui_simbolo_especial = any(
            caractere in ativo
            for caractere in (".", "^", "=", "-", "/")
        )

        if not possui_simbolo_especial:
            ativo = f"{ativo}{sufixo_b3}"

        if ativo not in resultado:
            resultado.append(ativo)

    if len(resultado) < 2:
        raise ValueError("Informe pelo menos dois ativos diferentes.")

    return resultado


ativos = normalizar_ativos(ATIVOS)
print("Ativos selecionados:", ", ".join(ativos))


## 4. Download dos preços ajustados

O download usa preços ajustados por proventos e desdobramentos.

A implementação também:

- solicita todos os ativos em uma única chamada;
- tenta reparar anomalias conhecidas de preço;
- aceita a estrutura multinível atual do `yfinance`;
- remove fusos horários para facilitar a manipulação;
- identifica ativos sem dados;
- preserva apenas o período comum para uma comparação justa.


In [ ]:
def extrair_fechamento(dados: pd.DataFrame) -> pd.DataFrame:
    """Extrai a coluna Close de retornos simples ou multinível."""

    if dados.empty:
        raise ValueError("O Yahoo Finance não devolveu dados.")

    if isinstance(dados.columns, pd.MultiIndex):
        nivel_0 = dados.columns.get_level_values(0)
        nivel_1 = dados.columns.get_level_values(1)

        if "Close" in nivel_0:
            fechamento = dados.xs("Close", axis=1, level=0)
        elif "Close" in nivel_1:
            fechamento = dados.xs("Close", axis=1, level=1)
        else:
            raise ValueError(
                "A resposta não contém o campo Close esperado."
            )
    else:
        if "Close" not in dados.columns:
            raise ValueError(
                "A resposta não contém a coluna Close esperada."
            )
        fechamento = dados[["Close"]].copy()

    if isinstance(fechamento, pd.Series):
        fechamento = fechamento.to_frame()

    fechamento.columns = [
        str(coluna).strip().upper()
        for coluna in fechamento.columns
    ]

    return fechamento


def baixar_precos_ajustados(
    ativos: list[str],
    data_inicio: str,
    data_fim: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Baixa preços ajustados e devolve base completa e base comum."""

    try:
        if hasattr(yf, "config"):
            yf.config.network.retries = 2

        dados = yf.download(
            tickers=ativos,
            start=data_inicio,
            end=data_fim,
            interval="1d",
            auto_adjust=True,
            repair=True,
            actions=False,
            threads=True,
            progress=False,
            timeout=30,
            group_by="column",
            multi_level_index=True,
        )
    except Exception as erro:
        raise RuntimeError(
            "Não foi possível baixar os dados do Yahoo Finance."
        ) from erro

    precos = extrair_fechamento(dados)
    precos = precos.sort_index()
    precos = precos.loc[~precos.index.duplicated(keep="first")]

    if getattr(precos.index, "tz", None) is not None:
        precos.index = precos.index.tz_localize(None)

    precos = precos.dropna(axis=1, how="all")
    precos.index.name = "data"

    ativos_sem_dados = [
        ativo for ativo in ativos
        if ativo not in precos.columns
    ]

    if ativos_sem_dados:
        print(
            "Ativos ignorados por falta de dados:",
            ", ".join(ativos_sem_dados),
        )

    if precos.shape[1] < 2:
        raise ValueError(
            "Menos de dois ativos possuem dados válidos."
        )

    precos_comuns = precos.dropna(how="any")

    if precos_comuns.empty:
        raise ValueError(
            "Os ativos não possuem um período comum com dados completos."
        )

    return precos, precos_comuns


precos_completos, precos = baixar_precos_ajustados(
    ativos=ativos,
    data_inicio=DATA_INICIO,
    data_fim=DATA_FIM,
)


## 5. Cobertura dos dados

Antes de calcular os indicadores, é importante conhecer:

- o primeiro e o último registro disponível para cada ativo;
- a quantidade de observações;
- o período comum efetivamente usado na comparação.


In [ ]:
def primeiro_indice_valido(serie: pd.Series):
    return serie.first_valid_index()


def ultimo_indice_valido(serie: pd.Series):
    return serie.last_valid_index()


cobertura = pd.DataFrame({
    "primeira_data": precos_completos.apply(primeiro_indice_valido),
    "ultima_data": precos_completos.apply(ultimo_indice_valido),
    "observacoes": precos_completos.notna().sum(),
})

display(cobertura)

print(
    "Período comum utilizado:",
    precos.index.min().date(),
    "até",
    precos.index.max().date(),
)
print("Pregões utilizados:", len(precos))

display(precos.head())


## 6. Retornos e indicadores

Os retornos diários são calculados sem preenchimento automático de valores ausentes.

### Indicadores

- **Retorno total:** variação acumulada entre o início e o fim.
- **CAGR:** retorno anualizado composto.
- **Volatilidade:** desvio-padrão anualizado dos retornos.
- **Sharpe:** retorno excedente por unidade de volatilidade.
- **Sortino:** retorno excedente dividido apenas pelo risco de queda.
- **Máximo drawdown:** maior perda desde um pico anterior.
- **Calmar:** CAGR dividido pelo valor absoluto do máximo drawdown.
- **Dias positivos:** percentual de pregões com retorno acima de zero.


In [ ]:
def calcular_indicadores(
    precos: pd.DataFrame,
    taxa_livre_risco_anual: float = 0.0,
    periodos_ano: int = 252,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Calcula retornos, drawdowns e métricas por ativo."""

    retornos = precos.pct_change(fill_method=None).dropna(how="any")

    if retornos.empty:
        raise ValueError(
            "Não há observações suficientes para calcular retornos."
        )

    quantidade_anos = len(retornos) / periodos_ano
    retorno_total = precos.iloc[-1] / precos.iloc[0] - 1
    cagr = (1 + retorno_total) ** (1 / quantidade_anos) - 1

    volatilidade = retornos.std(ddof=1) * np.sqrt(periodos_ano)

    taxa_livre_risco_diaria = (
        (1 + taxa_livre_risco_anual) ** (1 / periodos_ano) - 1
    )
    retornos_excedentes = retornos - taxa_livre_risco_diaria

    sharpe = (
        retornos_excedentes.mean()
        / retornos.std(ddof=1).replace(0, np.nan)
        * np.sqrt(periodos_ano)
    )

    retornos_negativos = retornos.where(retornos < 0)
    desvio_negativo = (
        retornos_negativos.std(ddof=1)
        * np.sqrt(periodos_ano)
    )
    retorno_excedente_anual = (
        retornos_excedentes.mean() * periodos_ano
    )
    sortino = (
        retorno_excedente_anual
        / desvio_negativo.replace(0, np.nan)
    )

    drawdowns = precos / precos.cummax() - 1
    max_drawdown = drawdowns.min()

    calmar = cagr / max_drawdown.abs().replace(0, np.nan)

    indicadores = pd.DataFrame({
        "retorno_total_pct": retorno_total * 100,
        "cagr_pct": cagr * 100,
        "volatilidade_anual_pct": volatilidade * 100,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown_pct": max_drawdown * 100,
        "calmar": calmar,
        "melhor_dia_pct": retornos.max() * 100,
        "pior_dia_pct": retornos.min() * 100,
        "dias_positivos_pct": (retornos > 0).mean() * 100,
    })

    indicadores.index.name = "ativo"

    return indicadores, retornos, drawdowns


indicadores, retornos, drawdowns = calcular_indicadores(
    precos=precos,
    taxa_livre_risco_anual=TAXA_LIVRE_RISCO_ANUAL,
    periodos_ano=DIAS_NEGOCIACAO_ANO,
)

display(indicadores.round(2))


## 7. Evolução do investimento inicial

Cada série é normalizada para começar com o mesmo valor. Assim, é possível comparar quanto o investimento teria se transformado em cada ativo.


In [ ]:
investimento = (
    precos
    .div(precos.iloc[0])
    .mul(VALOR_INICIAL)
)

ax = investimento.plot(figsize=(13, 6), linewidth=2)

ax.set_title(
    f"Evolução de um investimento inicial de R$ {VALOR_INICIAL:,.2f}"
)
ax.set_xlabel("Data")
ax.set_ylabel("Valor acumulado (R$)")
ax.legend(title="Ativo")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Drawdown

O drawdown mede a queda percentual de um ativo em relação ao maior preço observado até aquele momento.


In [ ]:
ax = (drawdowns * 100).plot(
    figsize=(13, 6),
    linewidth=1.5,
)

ax.set_title("Drawdown dos ativos")
ax.set_xlabel("Data")
ax.set_ylabel("Drawdown (%)")
ax.legend(title="Ativo")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Correlação dos retornos

A correlação varia de `-1` a `1`:

- próxima de `1`: os ativos tendem a se mover na mesma direção;
- próxima de `0`: pouca relação linear;
- próxima de `-1`: movimentos opostos.


In [ ]:
correlacao = retornos.corr()

fig, ax = plt.subplots(figsize=(8, 6))
imagem = ax.imshow(
    correlacao,
    vmin=-1,
    vmax=1,
    aspect="auto",
)

fig.colorbar(imagem, ax=ax, label="Correlação")

ax.set_xticks(range(len(correlacao.columns)))
ax.set_xticklabels(correlacao.columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlacao.index)))
ax.set_yticklabels(correlacao.index)

for linha in range(len(correlacao.index)):
    for coluna in range(len(correlacao.columns)):
        ax.text(
            coluna,
            linha,
            f"{correlacao.iloc[linha, coluna]:.2f}",
            ha="center",
            va="center",
        )

ax.set_title("Correlação entre os retornos diários")

plt.tight_layout()
plt.show()

display(correlacao.round(2))


## 10. Retornos anuais

A tabela abaixo resume o retorno de cada ativo por ano civil. O primeiro ano pode representar um período parcial, dependendo da data inicial configurada.


In [ ]:
precos_anuais = precos.resample("YE").last()

retornos_anuais = (
    precos_anuais
    .pct_change(fill_method=None)
    .mul(100)
)

retornos_anuais.index = retornos_anuais.index.year
retornos_anuais.index.name = "ano"

display(retornos_anuais.round(2))


## 11. Rankings

São apresentados dois rankings:

1. retorno total;
2. índice de Sharpe, que considera o retorno em relação à volatilidade.


In [ ]:
ranking_retorno = indicadores.sort_values(
    "retorno_total_pct",
    ascending=False,
)

ranking_sharpe = indicadores.sort_values(
    "sharpe",
    ascending=False,
)

print("Ranking por retorno total")
display(ranking_retorno.round(2))

print("Ranking por índice de Sharpe")
display(ranking_sharpe.round(2))


## 12. Salvamento opcional

Quando `SALVAR_RESULTADOS = True`, o notebook salva as principais tabelas e recria os gráficos dentro de `reports/`.


In [ ]:
def salvar_resultados(
    indicadores: pd.DataFrame,
    correlacao: pd.DataFrame,
    retornos_anuais: pd.DataFrame,
    investimento: pd.DataFrame,
    drawdowns: pd.DataFrame,
    pasta_tabelas: Path,
    pasta_figuras: Path,
) -> None:
    pasta_tabelas.mkdir(parents=True, exist_ok=True)
    pasta_figuras.mkdir(parents=True, exist_ok=True)

    indicadores.to_csv(
        pasta_tabelas / "indicadores.csv",
        encoding="utf-8-sig",
    )
    correlacao.to_csv(
        pasta_tabelas / "correlacao.csv",
        encoding="utf-8-sig",
    )
    retornos_anuais.to_csv(
        pasta_tabelas / "retornos_anuais.csv",
        encoding="utf-8-sig",
    )

    ax = investimento.plot(figsize=(13, 6), linewidth=2)
    ax.set_title("Evolução do investimento inicial")
    ax.set_xlabel("Data")
    ax.set_ylabel("Valor acumulado (R$)")
    ax.legend(title="Ativo")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        pasta_figuras / "evolucao_investimento.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.close()

    ax = (drawdowns * 100).plot(figsize=(13, 6))
    ax.set_title("Drawdown dos ativos")
    ax.set_xlabel("Data")
    ax.set_ylabel("Drawdown (%)")
    ax.legend(title="Ativo")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        pasta_figuras / "drawdowns.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.close()

    print("Resultados salvos em reports/.")


if SALVAR_RESULTADOS:
    salvar_resultados(
        indicadores=indicadores,
        correlacao=correlacao,
        retornos_anuais=retornos_anuais,
        investimento=investimento,
        drawdowns=drawdowns,
        pasta_tabelas=PASTA_TABELAS,
        pasta_figuras=PASTA_FIGURAS,
    )
else:
    print(
        "Salvamento desativado. "
        "Defina SALVAR_RESULTADOS = True para gerar os arquivos."
    )


## 13. Resumo automático

Os destaques abaixo são calculados com base nos dados obtidos no momento da execução.


In [ ]:
melhor_retorno = indicadores["retorno_total_pct"].idxmax()
melhor_sharpe = indicadores["sharpe"].idxmax()
menor_volatilidade = indicadores[
    "volatilidade_anual_pct"
].idxmin()
menor_drawdown = indicadores[
    "max_drawdown_pct"
].idxmax()

print(
    "Maior retorno total:",
    melhor_retorno,
    f"({indicadores.loc[melhor_retorno, 'retorno_total_pct']:.2f}%).",
)
print(
    "Melhor índice de Sharpe:",
    melhor_sharpe,
    f"({indicadores.loc[melhor_sharpe, 'sharpe']:.2f}).",
)
print(
    "Menor volatilidade anualizada:",
    menor_volatilidade,
    f"({indicadores.loc[menor_volatilidade, 'volatilidade_anual_pct']:.2f}%).",
)
print(
    "Menor perda máxima:",
    menor_drawdown,
    f"({indicadores.loc[menor_drawdown, 'max_drawdown_pct']:.2f}%).",
)


## 14. Conclusão e limitações

O projeto permite comparar ativos sob diferentes perspectivas. Um ativo com maior retorno não necessariamente possui:

- menor volatilidade;
- melhor índice de Sharpe;
- menor drawdown;
- menor correlação com os demais.

Por isso, a análise conjunta de retorno, risco e correlação é mais informativa do que um ranking baseado apenas no retorno acumulado.

### Limitações

- os dados são obtidos de um serviço externo e podem sofrer alterações;
- custos de negociação e impostos não são considerados;
- a taxa livre de risco é configurada manualmente;
- preços ajustados não representam uma operação real;
- ativos com históricos diferentes são comparados no período comum;
- resultados históricos não garantem desempenho futuro.

> Este notebook é um projeto educacional e não constitui recomendação de investimento.
